# Stencil Computation for 2D Gray-Scott Using Julia



# 1. Installation and Setup



JuliaGPU packages are easy to install: Just do `Pkg.add("Package Name")` to install the package.



In [ ]:
using Pkg
Pkg.add(["BenchmarkTools", "ProgressMeter", "Plots", "ColorTypes", "KernelAbstractions", "CUDA", "AMDGPU", "Metal"])



In [ ]:
using BenchmarkTools, ProgressMeter, Plots, ColorTypes, KernelAbstractions

In [ ]:
Pkg.status()



## 1.1. Selecting a backend

GPU backend packages provide bindings to NVIDIA's CUDA or respective other vendor. CUDA.jl provides all of the compiler and runtime logic needed to program NVIDIA GPUs; the only thing you need to provide is a functional NVIDIA driver (which most HPC systems already have installed and configured), but you don't need to install the CUDA toolkit! CUDA.jl downloads one if it's not already available on your system (again, HPC systems usually have this provided for you):



In [ ]:
# add all options for different backends and array types
DEV = :NVIDIA

if DEV == :NVIDIA
    using CUDA
    ArrayKA = CUDA.CuArray
    Backend = CUDA.CUDABackend()
elseif DEV == :AMD
    using AMDGPU
    ArrayKA = AMDGPU.ROCArray
    Backend = AMDGPU.ROCBackend()
elseif DEV == :Metal
    using Metal
    ArrayKA = Metal.MtlArray
    Backend = Metal.MetalBackend()
else DEV == :CPU
    ArrayKA = Array
    Backend = CPU()
end



# 2. Gray-Scott Reaction-Diffusion Model

The Gray-Scott Reaction-Diffusion model is a mathematical model used to describe how two interacting substances, typically denoted as $U$ and $V$, evolve over time in a spatial domain. This model captures the combined effects of chemical reactions and diffusion, leading to complex pattern formations such as spots, stripes, and waves, which resemble natural phenomena like animal skin patterns or chemical reactions in a petri dish.

## 2.1. Equations

The evolution of $ U $ and $ V $ over time is described by the following partial differential equations (PDEs):

$$
\frac{\partial U}{\partial t} = D_u \nabla^2 U - UV^2 + F(1 - U)
$$

$$
\frac{\partial V}{\partial t} = D_v \nabla^2 V + UV^2 - (F + k)V
$$

- **$D_u$**: Diffusion coefficient of $U$, controlling its spread rate.
- **$D_v$**: Diffusion coefficient of $V$, controlling its spread rate.
- **$F$**: Feed rate of  $U$, representing the replenishment of  $U$ in the system.
- **$ k $**: Kill rate of $V$, representing the removal of $V$ from the system.
- **$ \nabla^2 $**: Laplacian operator, representing the diffusion term.

## 2.2. 2D 5-Point Stencil

The 2D 5-point stencil is used to compute the diffusion terms ($ D_u \nabla^2 U $ and $ D_v \nabla^2 V $) in the Gray-Scott reaction-diffusion equations. It helps model how the substances $ U $ and $V $ spread through the 2D spatial domain by capturing the local flow of concentration between neighboring points.

![Input grid, 5-point stencil operator, and output grid](img/stencil_image1.png)

The left grid is the input field. The center shows the five cells the stencil touches at $(i, j)$: the point itself and its four orthogonal neighbors. The right grid is where the updated value is written. Every interior point of $U$ and $V$ is updated this way each time step.

![5-point stencil $S_{i,j}$ centered on $v_{i,j}$ with four neighbors](img/stencil_image2.webp)

On the mesh, $S_{i,j}$ is that same cross: the black center $v_{i,j}$ and the four yellow neighbors. Points on the domain edge are missing a neighbor, so the codes below skip the boundary (ghost) cells and only apply the stencil for $2 \le x \le n-1$ and $2 \le y \le n-1$.

#### Mathematical Formulation

The Laplacian of a function $ u $ at point $(i, j)$ using the 2D 5-point stencil is approximated as:

$$
\nabla^2 u_{i, j} \approx \frac{1}{h^2} \left( u_{i-1, j} + u_{i+1, j} - 2 \cdot u_{i, j} \right) + \frac{1}{h^2} \left( u_{i, j-1} + u_{i, j+1} - 2 \cdot u_{i, j} \right)
$$

- **$ h $** is the grid spacing (`dx` in the code).
- Each term is the centered second difference along one axis; adding them gives $\nabla^2 u$ from the five stencil points above.



# 3. From CPU to KernelAbstractions

Start from the reference CPU implementation, read the GPU exercice, then fill in the code in §3.3. A complete KA solution is in `proposed-solution/gray-scott.ipynb`.



## 3.1. Reference CPU implementation

Run this first so you have a correct baseline. You will port `calculate_gs_cpu!` / `simulate_gs_cpu` to KernelAbstractions in §3.3.



In [ ]:
# Reference CPU implementation — start from this, then port it to KernelAbstractions below.

function laplacian(A, x, y, dx)
    @inbounds DAx = (A[x-1, y] - (2 * A[x, y]) + A[x+1, y]) / dx
    @inbounds DAy = (A[x, y-1] - (2 * A[x, y]) + A[x, y+1]) / dx
    return DAx + DAy
end

function calculate_gs_cpu!(U, V, D_u, D_v, F, k, dt, dx, nx, ny, noise)
    ΔU = zeros(nx, ny)
    ΔV = zeros(nx, ny)

    for x in 2:(nx-1), y in 2:(ny-1)
        @inbounds ΔU[x, y] = laplacian(U, x, y, dx)
        @inbounds ΔV[x, y] = laplacian(V, x, y, dx)
    end

    UVV = U .* V .* V
    N = noise .* randn(nx, ny)
    U .+= (D_u .* ΔU .- UVV .+ F .* (1 .- U) .+ N) .* dt
    V .+= (D_v .* ΔV .+ UVV .- (F .+ k) .* V) .* dt
    return
end

function simulate_gs_cpu(n = 500 #=Grid size=#, nsteps = 10000 #=Number of iterations=# ;
                         init_multi = 0.5,
                         noise = 0.0,
                         D_u = 0.16,
                         D_v = 0.08,
                         F = 0.035,
                         k = 0.06,
                         dt = 1.0,
                         dx = 1.0,
                         plot_every = 500)

    U = rand(0.0:0.001:0.1, n, n)
    V = zeros(n, n)

    n_mid = div(n, 2)
    U[n_mid-5:n_mid+5, n_mid-5:n_mid+5] .= 1.0 .* init_multi
    V[n_mid-5:n_mid+5, n_mid-5:n_mid+5] .= 0.5 .* init_multi

    prog = Progress(div(nsteps, plot_every))
    plt = Plots.plot(title="Gray-Scott Model (CPU)", xlabel="X", ylabel="Y", size=(600, 600))

    anim = Plots.@animate for T in 1:div(nsteps, plot_every)
        for t in 1:plot_every
            calculate_gs_cpu!(U, V, D_u, D_v, F, k, dt, dx, n, n, noise)
        end
        next!(prog)
        Plots.heatmap!(plt, U, color=cgrad([:black, :blue, :white]), clims=extrema(U), title="Gray-Scott Model (CPU)")
    end
    gif(anim, "gs_cpu_serial.gif"; fps=10)
end



In [ ]:
@time simulate_gs_cpu(500, 10000)



## 3.2. Exercice: GPU implementation

**Goal.** Port the reference CPU Gray-Scott from §3.1 to KernelAbstractions so it runs on the `Backend` from §1.1. In §3.3 you will fill the `TODO`s; a complete solution is in `proposed-solution/gray-scott.ipynb`.

**Already done for you.** Keep `laplacian`, the host init / seed / progress bar / heatmap loop, and the PDE math. Those match the CPU version and are filled in — do not turn them into exercices.

Work through the tasks below. Each one says what to change and why it is required on the GPU.

1. **Turn the time step into a `@kernel`.**
   Replace `function calculate_gs_ka!(...)` with `@kernel function calculate_gs_ka!(...)`.
   *Why:* only `@kernel` functions are compiled for the GPU backend. A normal `function` stays on the host. You will not call it directly afterward — you build a launcher with `calculate_gs_ka!(Backend, workgroupsize)`.

2. **Replace the serial `for` loop with `@index` and a ghost-cell guard.**
   ```julia
   x, y = @index(Global, NTuple)[1:2]
   if (1 < x < nx) && (1 < y < ny)
       ...
   end
   ```
   *Why:* the GPU runs one work-item per grid point in parallel; there is no single thread walking the mesh. `@index(Global)` is that point’s coordinates. `ndrange=(n,n)` also launches edge threads, so the `if` prevents out-of-bounds stencil reads (same idea as `2:(n-1)` on the CPU).

3. **Allocate register memory up front with `@private` — do not use CPU scratch grids.**
   On the CPU, `ΔU = zeros(nx, ny)` is a whole-domain array in host RAM. In KernelAbstractions, declare per-thread **register** storage at the start of the kernel (before the index / `if`), the same way the matmul notebook uses `@private`:
   ```julia
   ΔU  = @private Float64 1
   ΔV  = @private Float64 1
   UVV = @private Float64 1
   Nr  = @private Float64 1
   DU  = @private Float64 1
   DV  = @private Float64 1
   ```
   Then store through `ΔU[1] = ...`, `DU[1] = ...`, and so on. `U` and `V` stay in **device global memory** (`KernelAbstractions.allocate`).
   *Why:* `@private` is how KA asks for per-work-item storage that lives in registers (fast memory next to the ALU). A kernel must not call `zeros(nx, ny)` on every thread. The CPU scratch arrays only fed a later broadcast; each work-item only needs a few register slots.

4. **Barrier around the in-place update with `@synchronize`.**
   After the register temporaries `DU[1]` / `DV[1]` are ready from global reads of `U` / `V`:
   ```julia
   @synchronize()
   @inbounds U[x, y] += DU[1] * dt   # write back to global memory
   @inbounds V[x, y] += DV[1] * dt
   @synchronize()
   ```
   *Why:* your stencil reads neighbors owned by other threads in the workgroup (still in global memory). Without a barrier, some threads may write global `U`/`V` before others finish reading, and the Laplacian mixes old and new values. The CPU avoided that by writing separate scratch arrays first. Also omit `return` — KA kernels do not allow it.

5. **Wire the driver to device global memory.**
   - `allocate` + `copyto!` host → device — *why:* the kernel cannot see host RAM; `U` / `V` must live in **device global memory** on `Backend`.
   - `kernel = calculate_gs_ka!(Backend, ...); kernel(...; ndrange=(n,n))` — *why:* specialize for the backend and launch one work-item per grid point (replaces `calculate_gs_cpu!(U, V, ...)`).
   - `copyto!` device → host + `synchronize` before plotting — *why:* `Plots` reads host memory, and without `synchronize` the heatmap can show stale global-memory data.



## 3.3. Now you: move it to GPU code

Follow the exercice in §3.2. Same-as-CPU pieces are already filled in; complete each `TODO`.



In [ ]:
# Port the reference CPU implementation (§3.2).
# Same-as-CPU logic is filled in; TODOs mark what you must change.
# Full answer: proposed-solution/gray-scott.ipynb

# TODO (1): use `@kernel function` here instead of `function`
function calculate_gs_ka!(U, V, D_u, D_v, F, k, dt, dx, nx, ny, noise)
    # TODO (3): allocate register memory in advance with `@private` (e.g. ΔU = @private Float64 1)
    # for ΔU, ΔV, UVV, Nr, DU, DV — then use ΔU[1], DU[1], … below

    # TODO (2): use `@index(Global, NTuple)` for (x, y), then an `if` to skip ghost cells
    # (replace this CPU-style loop)
    for x in 2:(nx-1), y in 2:(ny-1)

        # Same PDE as CPU, but into register slots (after you add `@private` above)
        ΔU[1] = laplacian(U, x, y, dx)
        ΔV[1] = laplacian(V, x, y, dx)

        UVV[1] = @inbounds U[x, y] * (V[x, y] ^ 2)
        Nr[1] = noise * randn(Float64)
        DU[1] = @inbounds D_u * ΔU[1] - UVV[1] + F * (1 - U[x, y]) + Nr[1]
        DV[1] = @inbounds D_v * ΔV[1] + UVV[1] - (F + k) * V[x, y]

        # TODO (4): use `@synchronize` before and after these global-memory writes
        @inbounds U[x, y] += DU[1] * dt
        @inbounds V[x, y] += DV[1] * dt
    end
end

function simulate_gs_ka(n = 500 #=Grid size=#, nsteps = 10000 #=Number of iterations=# ;
                        init_multi = 0.5,
                        noise = 0.0,
                        D_u = 0.16,
                        D_v = 0.08,
                        F = 0.035,
                        k = 0.06,
                        dt = 1.0,
                        dx = 1.0,
                        plot_every = 500)

    # Same host init as simulate_gs_cpu
    U = rand(0.0:0.001:0.1, n, n)
    V = zeros(n, n)

    n_mid = div(n, 2)
    U[n_mid-5:n_mid+5, n_mid-5:n_mid+5] .= 1.0 * init_multi
    V[n_mid-5:n_mid+5, n_mid-5:n_mid+5] .= 0.5 * init_multi

    # TODO (5): use `KernelAbstractions.allocate` + `copyto!` for device global memory (U_gpu, V_gpu)

    # TODO (5): use `calculate_gs_ka!(Backend, workgroupsize)` to build the kernel launcher

    # Same progress / plot setup as CPU
    prog = Progress(div(nsteps, plot_every))
    plt = Plots.plot(title="Gray-Scott Model", xlabel="X", ylabel="Y", size=(600, 600))

    anim = Plots.@animate for t in 1:div(nsteps, plot_every)
        for _ in 1:plot_every
            # TODO (5): launch the kernel on U_gpu, V_gpu with `ndrange=(n, n)`
        end

        # TODO (5): use `copyto!` back to host `U`, then `KernelAbstractions.synchronize(Backend)`

        # Same plotting as CPU
        next!(prog)
        Plots.heatmap!(plt, U, color=cgrad([:black, :blue, :white]), clims=extrema(U), title="Gray-Scott Model")
    end
    gif(anim, "gs_ka.gif"; fps=10)
end



In [ ]:
@time simulate_gs_ka(500, 10000)



# 4. Experiments

Try different noise levels and parameters once your KA implementation runs.



In [ ]:
# Add some noise
@time simulate_gs_ka(500, 10000; noise=0.01)



In [ ]:
# Add a lot more noise
@time simulate_gs_ka(500, 10000; noise=0.03)



In [ ]:
# Tweak some parameters
@time simulate_gs_ka(500, 50000; k=.06054, F=.07528, noise=0.0001, init_multi=1.0, plot_every=2000)

